# 03 - Traffic Density Generation

Convert per-frame vehicle counts from `data/processed/vehicle_counts.csv` into a time-series dataset for traffic prediction modelling:

1. Aggregate vehicle counts into fixed time intervals
2. Add temporal features (time of day, day of week)
3. Add lag features (t-1, t-2, t-3)
4. Apply smoothing / noise reduction
5. Compute rate of change, peak density, rolling mean, rolling standard deviation
6. Categorise traffic density into low / moderate / high

Output: `data/processed/traffic_density.csv`

**Split:** S01, S03, S04 = train · S02 = validation · S05 = test

In [18]:
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd

In [19]:
processed_dir = Path("../data/processed")
input_path = processed_dir / "vehicle_counts.csv"
output_path = processed_dir / "traffic_density.csv"

interval_seconds = 15          # aggregation window for the time-series
lag_steps = [1, 2]          # t-1, t-2, t-3 lag features
smoothing_window = 3           # rolling window used to smooth raw counts
rolling_window = 3             # rolling window for rolling mean / std / peak features


# quantile thresholds for low / moderate / high traffic categorisation, computed from the train split only
density_quantiles = [0.3, 0.6]

In [20]:
vehicle_counts = pd.read_csv(input_path)
vehicle_counts = vehicle_counts.sort_values(["scene_id", "camera_id", "timestamp_sec"])
print(f"Loaded {len(vehicle_counts)} frame-level rows from {input_path}")
vehicle_counts.head()

Loaded 11721 frame-level rows from ..\data\processed\vehicle_counts.csv


,scene_id,camera_id,split,frame_file,source_frame_idx,timestamp_sec,vehicle_count
0,S01,c001,train,frame_000000.jpg,0,0.0,1
1,S01,c001,train,frame_000001.jpg,10,1.0,0
2,S01,c001,train,frame_000002.jpg,20,2.0,1
3,S01,c001,train,frame_000003.jpg,30,3.0,2
4,S01,c001,train,frame_000004.jpg,40,4.0,1


In [21]:
def aggregate_intervals(df: pd.DataFrame, interval_seconds: int) -> pd.DataFrame:
    df = df.copy()
    df["interval_id"] = (df["timestamp_sec"] // interval_seconds).astype(int)

    aggregated = (
        df.groupby(["scene_id", "camera_id", "split", "interval_id"])
        .agg(vehicle_count=("vehicle_count", "mean"), n_frames=("vehicle_count", "count"))
        .reset_index()
    )
    aggregated["interval_start_sec"] = aggregated["interval_id"] * interval_seconds
    return aggregated.sort_values(["scene_id", "camera_id", "interval_start_sec"]).reset_index(drop=True)

In [22]:
density = aggregate_intervals(vehicle_counts, interval_seconds)
print(f"Aggregated into {len(density)} interval-level rows ({interval_seconds}s intervals)")
density.head()

Aggregated into 815 interval-level rows (15s intervals)


,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec
0,S01,c001,train,0,2.200000,15,0
1,S01,c001,train,1,1.133333,15,15
2,S01,c001,train,2,1.600000,15,30
3,S01,c001,train,3,3.400000,15,45
4,S01,c001,train,4,3.333333,15,60


In [23]:
def add_temporal_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["elapsed_time_sec"] = df["interval_start_sec"]

    clip_duration = df.groupby(["scene_id", "camera_id"])["interval_start_sec"].transform("max")
    df["relative_position"] = np.where(clip_duration > 0, df["elapsed_time_sec"] / clip_duration, 0.0)
    return df

In [24]:
density = add_temporal_features(density)
density.head()

,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position
0,S01,c001,train,0,2.200000,15,0,0,0.000000
1,S01,c001,train,1,1.133333,15,15,15,0.076923
2,S01,c001,train,2,1.600000,15,30,30,0.153846
3,S01,c001,train,3,3.400000,15,45,45,0.230769
4,S01,c001,train,4,3.333333,15,60,60,0.307692


In [25]:
def add_smoothing(df: pd.DataFrame, smoothing_window: int) -> pd.DataFrame:
    df = df.copy()
    df["vehicle_count_smoothed"] = (
        df.groupby(["scene_id", "camera_id"])["vehicle_count"]
        .transform(lambda s: s.rolling(window=smoothing_window, min_periods=1).mean())
    )
    return df

In [26]:
density = add_smoothing(density, smoothing_window)
density.head()

,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed
0,S01,c001,train,0,2.200000,15,0,0,0.000000,2.200000
1,S01,c001,train,1,1.133333,15,15,15,0.076923,1.666667
2,S01,c001,train,2,1.600000,15,30,30,0.153846,1.644444
3,S01,c001,train,3,3.400000,15,45,45,0.230769,2.044444
4,S01,c001,train,4,3.333333,15,60,60,0.307692,2.777778


In [27]:
def add_lag_features(df: pd.DataFrame, lag_steps: list) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby(["scene_id", "camera_id"])["vehicle_count_smoothed"]
    for lag in lag_steps:
        df[f"lag_{lag}"] = grouped.shift(lag)
    return df

In [28]:
density = add_lag_features(density, lag_steps)
density.head()

,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed,lag_1,lag_2
0,S01,c001,train,0,2.200000,15,0,0,0.000000,2.200000,NaN,NaN
1,S01,c001,train,1,1.133333,15,15,15,0.076923,1.666667,2.200000,NaN
2,S01,c001,train,2,1.600000,15,30,30,0.153846,1.644444,1.666667,2.200000
3,S01,c001,train,3,3.400000,15,45,45,0.230769,2.044444,1.644444,1.666667
4,S01,c001,train,4,3.333333,15,60,60,0.307692,2.777778,2.044444,1.644444


In [29]:
def add_derived_features(df: pd.DataFrame, rolling_window: int) -> pd.DataFrame:
    df = df.copy()
    grouped = df.groupby(["scene_id", "camera_id"])["vehicle_count_smoothed"]

    df["rate_of_change"] = grouped.transform(lambda s: s.diff())
    df["peak_density"] = grouped.transform(lambda s: s.rolling(window=rolling_window, min_periods=1).max())
    df["rolling_mean"] = grouped.transform(lambda s: s.rolling(window=rolling_window, min_periods=1).mean())
    df["rolling_std"] = grouped.transform(lambda s: s.rolling(window=rolling_window, min_periods=1).std())
    df["rolling_std"] = df["rolling_std"].fillna(0.0)
    return df

In [30]:
density = add_derived_features(density, rolling_window)
density.head()

,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed,lag_1,lag_2,rate_of_change,peak_density,rolling_mean,rolling_std
0,S01,c001,train,0,2.200000,15,0,0,0.000000,2.200000,NaN,NaN,NaN,2.200000,2.200000,0.000000
1,S01,c001,train,1,1.133333,15,15,15,0.076923,1.666667,2.200000,NaN,-0.533333,2.200000,1.933333,0.377124
2,S01,c001,train,2,1.600000,15,30,30,0.153846,1.644444,1.666667,2.200000,-0.022222,2.200000,1.837037,0.314531
3,S01,c001,train,3,3.400000,15,45,45,0.230769,2.044444,1.644444,1.666667,0.400000,2.044444,1.785185,0.224800
4,S01,c001,train,4,3.333333,15,60,60,0.307692,2.777778,2.044444,1.644444,0.733333,2.777778,2.155556,0.574779


In [31]:
def compute_density_thresholds(df: pd.DataFrame, density_quantiles: list):
    train_values = df.loc[df["split"] == "train", "vehicle_count_smoothed"]
    low_cut, high_cut = train_values.quantile(density_quantiles)
    return low_cut, high_cut

In [32]:
def categorise_density(df: pd.DataFrame, low_cut: float, high_cut: float) -> pd.DataFrame:
    df = df.copy()

    def label(value):
        if value <= low_cut:
            return "low"
        elif value <= high_cut:
            return "moderate"
        return "high"

    df["density_category"] = df["vehicle_count_smoothed"].apply(label)
    return df

In [33]:
low_cut, high_cut = compute_density_thresholds(density, density_quantiles)
density = categorise_density(density, low_cut, high_cut)

print(f"Density thresholds (from train split): low <= {low_cut:.2f} < moderate <= {high_cut:.2f} < high")
density["density_category"].value_counts()

Density thresholds (from train split): low <= 2.42 < moderate <= 3.47 < high


density_category
high        410
low         282
moderate    123
Name: count, dtype: int64

In [34]:
density.to_csv(output_path, index=False)
print(f"Saved {len(density)} rows to {output_path}")
density.head()

Saved 815 rows to ..\data\processed\traffic_density.csv


,scene_id,camera_id,split,interval_id,vehicle_count,n_frames,interval_start_sec,elapsed_time_sec,relative_position,vehicle_count_smoothed,lag_1,lag_2,rate_of_change,peak_density,rolling_mean,rolling_std,density_category
0,S01,c001,train,0,2.200000,15,0,0,0.000000,2.200000,NaN,NaN,NaN,2.200000,2.200000,0.000000,low
1,S01,c001,train,1,1.133333,15,15,15,0.076923,1.666667,2.200000,NaN,-0.533333,2.200000,1.933333,0.377124,low
2,S01,c001,train,2,1.600000,15,30,30,0.153846,1.644444,1.666667,2.200000,-0.022222,2.200000,1.837037,0.314531,low
3,S01,c001,train,3,3.400000,15,45,45,0.230769,2.044444,1.644444,1.666667,0.400000,2.044444,1.785185,0.224800,low
4,S01,c001,train,4,3.333333,15,60,60,0.307692,2.777778,2.044444,1.644444,0.733333,2.777778,2.155556,0.574779,moderate
